In [12]:
import os
import requests
from dotenv import load_dotenv
from pprint import pprint
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langsmith import utils, Client, traceable
from langsmith.schemas import TracerSessionResult

load_dotenv()
print(f"LANGSMITH_WORKSPACE_ID: {os.getenv('LANGSMITH_WORKSPACE_ID')}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT')}")
print(f"LANGSMITH_TRACING: {utils.tracing_is_enabled()}")
langSmithClient = Client()

LANGSMITH_WORKSPACE_ID: da80a42a-4c73-47f1-91e7-0922c5222593
LANGSMITH_PROJECT: lca-lc-foundation
LANGSMITH_TRACING: True


# Get all LangSmith Workspace ID

In case you don't know your workspace ID, you can use the following code to get it. Make sure to set your `LANGSMITH_API_KEY` in the `.env` file before running this code.

In [6]:
response = requests.get(
    url="https://api.smith.langchain.com/api/v1/workspaces",
    headers={"x-api-key": os.getenv("LANGSMITH_API_KEY")}
)
for ws in response.json():
    print(f"Name: {ws['display_name']}, ID: {ws['id']}")

Name: Workspace 1, ID: da80a42a-4c73-47f1-91e7-0922c5222593


In [10]:
projects: list[TracerSessionResult] = list(langSmithClient.list_projects())
print(f"Total projects: {len(projects)}")

for project in projects:
    print(f"Project ID: {project.id}, Name: {project.name}")

Total projects: 1
Project ID: 895919fb-918b-4ef8-be43-a783d5e48d2e, Name: lca-lc-foundation


In [9]:
langSmithClient.create_project(project_name=os.getenv('LANGSMITH_PROJECT'), upsert=True)

TracerSession(id=UUID('895919fb-918b-4ef8-be43-a783d5e48d2e'), start_time=datetime.datetime(2026, 4, 9, 17, 47, 55, 454026, tzinfo=datetime.timezone.utc), end_time=None, description=None, name='lca-lc-foundation', extra=None, tenant_id=UUID('da80a42a-4c73-47f1-91e7-0922c5222593'), reference_dataset_id=None)

# Sample Traces

## 1. Traces with function call

In [ ]:
# run multiple traces into the same project
@traceable(project_name=os.getenv('LANGSMITH_PROJECT'))
def my_llm_call(query: str):
    return f"response to: {query}"

my_llm_call("hello")        # trace 1
my_llm_call("how are you")  # trace 2
my_llm_call("bye")          # trace 3

In [14]:
# verify multiple traces exist in the project
runs = list(langSmithClient.list_runs(project_name=os.getenv('LANGSMITH_PROJECT')))
print(f"Total traces in project: {len(runs)}")

for run in runs:
    print(f"Trace ID: {run.id}, Input: {run.inputs}")

Total traces in project: 3
Trace ID: 019d7361-9455-7481-bb99-1a9ae077b268, Input: {'query': 'bye'}
Trace ID: 019d7361-9455-7df2-8097-974af28ed0d3, Input: {'query': 'how are you'}
Trace ID: 019d7361-943b-7680-a83e-056ed0317314, Input: {'query': 'hello'}


## 2. Traces with agent execution

In [15]:
@tool("square_root", description="Calculate the square root of a number")
def square_root(x: float) -> float:
    return x ** 0.5

agent = create_agent(
    model="gpt-5-nano",
    tools=[square_root],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [ ]:
question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    input={"messages": [question]}
)

In [17]:
pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='6fe23796-5774-42d7-8d3d-a16543a2065a'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2135, 'prompt_tokens': 158, 'total_tokens': 2293, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2112, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DSo5xhiC4UuQ4AysPhOp3LqaWhVs8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d736e-91fc-7910-9b50-68dc075b1d7e-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_iG6EnHD6QFFRAfvm1nUOsGLs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 2135, 'total